# 02. CT 강도, mask 라벨과 표(annotation) 정합성

## 검사 이유

NIfTI의 저장 정수값은 `scl_slope`와 `scl_inter`를 적용해야 실제 값이 됩니다. 이를 생략하면 PhysioNet mask의 0/255 의미를 놓치거나 CT 강도 통계가 달라질 수 있습니다.

이 노트북은 사례별로 한 volume씩 읽어 메모리 사용을 제한합니다. 분위수는 최대 30만 voxel의 재현 가능한 간격 표본으로 계산하고, mask voxel 수는 정확하게 계산합니다.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd

def find_project_root(start=Path.cwd()):
    """VS Code가 프로젝트 폴더/노트북 폴더 중 어디를 cwd로 잡아도 동작하게 합니다."""
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "project_config.json").is_file() and (candidate / "ctqa_utils.py").is_file():
            return candidate
    raise FileNotFoundError("project_config.json이 있는 CT_Data_Quality_Notebooks 폴더를 VS Code에서 여세요.")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from ctqa_utils import *

CONFIG = json.loads((PROJECT_ROOT / "project_config.json").read_text(encoding="utf-8"))
DATA_ROOT = Path(CONFIG["data_root"])
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("프로젝트:", PROJECT_ROOT)
print("데이터:", DATA_ROOT)
print("결과 저장:", RESULTS)

프로젝트: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합
데이터: C:\Users\duddl\OneDrive\바탕 화면\CTdata
결과 저장: c:\workspace\hodgepodge\hy\데이터품질검증 및 통합\results


In [2]:
pair_path = RESULTS / "00_pair_manifest.csv"
if not pair_path.is_file():
    raise RuntimeError("먼저 00 노트북을 실행하세요.")
pairs = pd.read_csv(pair_path)

## 1. 사례별 강도와 라벨 검사

허용 라벨은 BHSD `{0,1,2,3,4,5}`, PhysioNet은 scaling 후 `{0,255}`입니다. 통합용 이진 mask는 두 source 모두 `mask > 0`으로 정의합니다.

In [3]:
expected_labels = {"BHSD": {0, 1, 2, 3, 4, 5}, "PhysioNet_CT_ICH": {0, 255}}
rows = []
errors = []
physionet_positive_slices = {}

for idx, item in pairs.iterrows():
    try:
        image_path, mask_path = Path(item.image_path), Path(item.mask_path)
        ih, mh = read_nifti_header(image_path), read_nifti_header(mask_path)
        ct = read_nifti_data(image_path, ih, scaled=True)
        mask = read_nifti_data(mask_path, mh, scaled=True)
        ct_stats = array_summary(ct, percentiles=(1, 50, 99))
        labels, counts = np.unique(mask, return_counts=True)
        rounded_labels = {int(round(float(x))) for x in labels}
        invalid = sorted(rounded_labels - expected_labels[item.source])
        binary_mask = mask > 0
        positive_voxels = int(binary_mask.sum())
        positive_by_slice = binary_mask.sum(axis=(0, 1)) > 0
        if item.source == "PhysioNet_CT_ICH":
            physionet_positive_slices[str(item.case_id).zfill(3)] = positive_by_slice.astype(np.uint8)

        rule = CONFIG["intensity_encoding_candidate"]
        encoding_candidate = bool(
            ct_stats["min"] >= rule["min_at_least"]
            and ct_stats["max"] <= rule["max_at_most"]
            and ct_stats["p99"] > rule["p99_above"]
        )
        rows.append({
            "source": item.source,
            "case_id": item.case_id,
            "ct_finite_fraction": ct_stats["finite_fraction"],
            "ct_min": ct_stats["min"],
            "ct_p01": ct_stats["p01"],
            "ct_p50": ct_stats["p50"],
            "ct_p99": ct_stats["p99"],
            "ct_max": ct_stats["max"],
            "ct_constant": bool(ct_stats["min"] == ct_stats["max"]),
            "ct_encoding_candidate": encoding_candidate,
            "mask_finite_fraction": float(np.isfinite(mask).mean()),
            "mask_labels_scaled": json.dumps(sorted(rounded_labels)),
            "invalid_labels": json.dumps(invalid),
            "invalid_label_count": len(invalid),
            "positive_voxels": positive_voxels,
            "positive_slices": int(positive_by_slice.sum()),
            "empty_mask": positive_voxels == 0,
            "small_positive_mask": 0 < positive_voxels < int(CONFIG["small_lesion_voxels"]),
            "mask_scl_slope": mh["scl_slope"],
            "mask_scl_inter": mh["scl_inter"],
        })
        del ct, mask, binary_mask
    except Exception as exc:
        errors.append({"source": item.source, "case_id": item.case_id, "error": repr(exc)})
    if (idx + 1) % 25 == 0:
        print(f"진행: {idx + 1}/{len(pairs)}")

values = pd.DataFrame(rows)
values.to_csv(RESULTS / "02_value_label_checks.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(errors).to_csv(RESULTS / "02_read_errors.csv", index=False, encoding="utf-8-sig")

value_summary = values.groupby("source").agg(
    cases=("case_id", "count"),
    nonfinite_cases=("ct_finite_fraction", lambda s: int((s < 1).sum())),
    constant_ct=("ct_constant", "sum"),
    encoding_candidates=("ct_encoding_candidate", "sum"),
    positive_masks=("empty_mask", lambda s: int((~s).sum())),
    empty_masks=("empty_mask", "sum"),
    small_positive_masks=("small_positive_mask", "sum"),
    invalid_label_cases=("invalid_label_count", lambda s: int((s > 0).sum())),
).reset_index()
print(value_summary.to_string(index=False))
print("읽기 오류:", len(errors))

진행: 25/267
진행: 50/267
진행: 75/267
진행: 100/267
진행: 125/267
진행: 150/267
진행: 175/267
진행: 200/267
진행: 225/267
진행: 250/267
          source  cases  nonfinite_cases  constant_ct  encoding_candidates  positive_masks  empty_masks  small_positive_masks  invalid_label_cases
            BHSD    192                0            0                    1             192            0                     4                    0
PhysioNet_CT_ICH     75                0            0                    0              36           39                     0                    0
읽기 오류: 0


## 2. PhysioNet slice-level CSV와 NIfTI mask 비교

CSV의 5개 출혈 subtype 중 하나라도 1이면 해당 slice의 mask도 양성이어야 합니다. 이 검사는 slice 순서가 뒤집혔거나 mask scaling을 빠뜨린 오류를 찾습니다.

In [4]:
csv_path = DATA_ROOT / "PhysioNet_CT_ICH" / "hemorrhage_diagnosis_raw_ct.csv"
labels_csv = pd.read_csv(csv_path)
subtype_cols = ["Intraventricular", "Intraparenchymal", "Subarachnoid", "Epidural", "Subdural"]
consistency_rows = []

for patient, group in labels_csv.groupby("PatientNumber", sort=True):
    case_id = str(int(patient)).zfill(3)
    group = group.sort_values("SliceNumber")
    csv_positive = (group[subtype_cols].sum(axis=1) > 0).to_numpy(dtype=np.uint8)
    mask_positive = physionet_positive_slices.get(case_id)
    same_count = mask_positive is not None and len(mask_positive) == len(csv_positive)
    mismatch = int(np.sum(mask_positive != csv_positive)) if same_count else None
    consistency_rows.append({
        "case_id": case_id,
        "csv_slices": len(csv_positive),
        "mask_slices": len(mask_positive) if mask_positive is not None else None,
        "slice_count_match": same_count,
        "positive_slice_mismatch": mismatch,
    })

consistency = pd.DataFrame(consistency_rows)
consistency.to_csv(RESULTS / "02_physionet_slice_consistency.csv", index=False, encoding="utf-8-sig")
print("환자 수:", len(consistency))
print("slice 수 불일치:", int((~consistency.slice_count_match).sum()))
print("양성 slice 불일치 합계:", int(consistency.positive_slice_mismatch.fillna(0).sum()))

환자 수: 75
slice 수 불일치: 0
양성 slice 불일치 합계: 0


## 3. 12-bit signed CT 인코딩 후보의 교정 전·후 비교

일부 CT가 0–4095 범위에 있으면서 높은 값에 공기가 몰리면 signed 12-bit 값이 unsigned로 저장됐을 가능성이 있습니다. 후보에만 `value > 2047 → value - 4096`을 적용해 분위수를 비교하고, 원본 메타데이터 확인 전에는 확정하지 않습니다.

In [5]:
candidates = values[values.ct_encoding_candidate]
correction_examples = []
for _, candidate in candidates.iterrows():
    item = pairs[(pairs.source == candidate.source) & (pairs.case_id == candidate.case_id)].iloc[0]
    ct = read_nifti_data(Path(item.image_path), scaled=True)
    corrected = np.where(ct > 2047, ct - 4096, ct)
    correction_examples.append({
        "source": candidate.source,
        "case_id": candidate.case_id,
        "before": array_summary(ct, percentiles=(1, 50, 99)),
        "after_candidate_transform": array_summary(corrected, percentiles=(1, 50, 99)),
    })
write_json(RESULTS / "02_intensity_correction_candidates.json", correction_examples)
print(json.dumps(correction_examples, ensure_ascii=False, indent=2))

[
  {
    "source": "BHSD",
    "case_id": "ID_a429fd9a_ID_2eabc43f3d",
    "before": {
      "finite_fraction": 1.0,
      "min": 0.0,
      "max": 4095.0,
      "p01": 11.0,
      "p50": 3096.0,
      "p99": 4078.0
    },
    "after_candidate_transform": {
      "finite_fraction": 1.0,
      "min": -2010.0,
      "max": 1994.0,
      "p01": -1000.0,
      "p50": -986.0,
      "p99": 1218.0
    }
  }
]


## 4. 단계 판정

- NaN/Inf, 상수 CT, 허용하지 않은 mask label은 0건이어야 합니다.
- PhysioNet의 빈 mask는 정상 음성 39건이므로 삭제하면 안 됩니다.
- 인코딩 후보는 원본 근거를 확인하고 교정 또는 격리해야 합니다.

In [6]:
step_result = {
    "read_errors": len(errors),
    "nonfinite_cases": int((values.ct_finite_fraction < 1).sum() + (values.mask_finite_fraction < 1).sum()),
    "constant_ct": int(values.ct_constant.sum()),
    "invalid_label_cases": int((values.invalid_label_count > 0).sum()),
    "intensity_encoding_candidates": int(values.ct_encoding_candidate.sum()),
    "physionet_slice_count_mismatch": int((~consistency.slice_count_match).sum()),
    "physionet_positive_slice_mismatch": int(consistency.positive_slice_mismatch.fillna(0).sum()),
}
write_json(RESULTS / "02_step_result.json", step_result)
print(step_result)

{'read_errors': 0, 'nonfinite_cases': 0, 'constant_ct': 0, 'invalid_label_cases': 0, 'intensity_encoding_candidates': 1, 'physionet_slice_count_mismatch': 0, 'physionet_positive_slice_mismatch': 0}
